# Emoji Classification (ResNet18 + file size feature)

PyTorch notebook for Colab. Uses ResNet18 pretrained on ImageNet and fuses a scalar feature: log(file size in bytes). Set `data_dir` to the folder containing `train/`, `test/`, and `train_labels.csv` (e.g., `/content/data`).

In [ ]:
import os, random, math, pathlib, json
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

# Config
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 64
VAL_RATIO = 0.2
EPOCHS = 6
LR = 1e-4
WEIGHT_DECAY = 1e-4
SMOKE_TEST = False  # set True for quick check (2 batches, 1 epoch)

data_dir = pathlib.Path("/content/data")
train_dir = data_dir / "train"
test_dir = data_dir / "test"
labels_csv = data_dir / "train_labels.csv"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

df = pd.read_csv(labels_csv)
label_names = sorted(df["Label"].unique())
label_to_idx = {l:i for i,l in enumerate(label_names)}
df["label_idx"] = df["Label"].map(label_to_idx)

from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=VAL_RATIO, stratify=df["Label"], random_state=SEED)
print(f"Train size: {len(train_df)}, Val size: {len(val_df)}, Classes: {len(label_names)}")

In [ ]:
# Dataset with log(file size) feature
class EmojiDataset(Dataset):
    def __init__(self, dataframe, image_dir, label_to_idx, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = pathlib.Path(image_dir)
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = str(row["Id"]).zfill(5) + ".png"
        label = self.label_to_idx[row["Label"]]
        path = self.image_dir / img_id
        size_bytes = os.path.getsize(path)
        size_feat = math.log1p(size_bytes)
        with Image.open(path) as img:
            img = img.convert('RGB')
        if self.transform:
            img = self.transform(img)
        return {
            "image": img,
            "label": label,
            "size_feat": torch.tensor(size_feat, dtype=torch.float32),
            "id": path.stem,
        }

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]
train_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ToTensor(),
    T.Normalize(mean=imagenet_mean, std=imagenet_std),
])
val_tfms = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=imagenet_mean, std=imagenet_std),
])

# Compute size normalization stats from train set
size_vals = [math.log1p(os.path.getsize(train_dir / (str(r.Id).zfill(5)+'.png'))) for r in train_df.itertuples()]
size_mean = np.mean(size_vals)
size_std = np.std(size_vals) + 1e-8
print('Size feature mean/std:', size_mean, size_std)

# Datasets/loaders
train_ds = EmojiDataset(train_df, train_dir, label_to_idx, transform=train_tfms)
val_ds = EmojiDataset(val_df, train_dir, label_to_idx, transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

if SMOKE_TEST:
    # take a couple of batches for a quick check
    train_loader = [next(iter(train_loader)) for _ in range(2)]
    val_loader = [next(iter(val_loader)) for _ in range(2)]
    EPOCHS = 1
    print('SMOKE_TEST enabled')

In [ ]:
# Model: ResNet18 + size fusion
class ResNet18WithSize(nn.Module):
    def __init__(self, num_classes, size_mean, size_std):
        super().__init__()
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        self.size_norm = nn.BatchNorm1d(1)
        # initialize batchnorm to approximate z-score
        with torch.no_grad():
            self.size_norm.weight.fill_(1.0 / size_std)
            self.size_norm.bias.fill_(-size_mean / size_std)
        self.classifier = nn.Linear(in_features + 1, num_classes)

    def forward(self, x, size_feat):
        feat = self.backbone(x)
        size_normed = self.size_norm(size_feat.unsqueeze(1))
        fused = torch.cat([feat, size_normed], dim=1)
        return self.classifier(fused)

model = ResNet18WithSize(num_classes=len(label_names), size_mean=size_mean, size_std=size_std).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_val_acc = 0.0
best_path = "/content/output/best_resnet18_size.pth"
os.makedirs('/content/output', exist_ok=True)

def run_one_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0
    total_correct = 0
    total = 0
    for batch in loader:
        images = batch["image"].to(device)
        labels = batch["label"].to(device)
        sizes = batch["size_feat"].to(device)
        with torch.set_grad_enabled(train):
            logits = model(images, sizes)
            loss = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, total_correct / total

for epoch in range(EPOCHS):
    train_loss, train_acc = run_one_epoch(train_loader, train=True)
    val_loss, val_acc = run_one_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}/{EPOCHS} | Train {train_loss:.4f}/{train_acc:.4f} | Val {val_loss:.4f}/{val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_path)
        print(f"Saved new best at {best_val_acc:.4f}")

print("Best val acc:", best_val_acc)
model.load_state_dict(torch.load(best_path, map_location=device))

In [ ]:
# Inference and submission
model.eval()
test_paths = sorted(test_dir.glob("*.png"))
if SMOKE_TEST:
    test_paths = test_paths[:64]

test_tfms = val_tfms

def load_test(path):
    with Image.open(path) as img:
        img = img.convert('RGB')
    return test_tfms(img), math.log1p(os.path.getsize(path)), path.stem

test_data = [load_test(p) for p in test_paths]
images = torch.stack([d[0] for d in test_data]).to(device)
sizes = torch.tensor([d[1] for d in test_data], dtype=torch.float32).to(device)

with torch.no_grad():
    logits = model(images, sizes)
    pred_idx = logits.argmax(1).cpu().tolist()
pred_labels = [label_names[i] for i in pred_idx]

submission = pd.DataFrame({
    "Id": [d[2] for d in test_data],
    "Label": pred_labels,
})
out_csv = "/content/output/submission.csv"
submission.to_csv(out_csv, index=False)
print("Saved submission to", out_csv, "rows:", len(submission))
if SMOKE_TEST:
    print("SMOKE_TEST was enabled (subset of test)")